<a href="https://colab.research.google.com/github/Merinroy010/CloudComputing/blob/master/osm_water.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade osmnx

In [ ]:
"""
Extract OpenStreetMap rivers and streams for Kerala, India
Requires: osmium, geopandas, shapely, requests
Install: pip install osmium geopandas shapely requests
"""

import requests
import geopandas as gpd
from shapely.geometry import LineString, Point
import json
import time

def download_osm_data(bbox, output_file="kerala_waterways.geojson"):
    """
    Download river and stream data from Overpass API for Kerala

    Parameters:
    bbox: tuple (min_lat, min_lon, max_lat, max_lon)
    output_file: output filename for GeoJSON
    """

    # Kerala bounding box (approximate)
    min_lat, min_lon, max_lat, max_lon = bbox

    # Overpass API query for waterways
    overpass_url = "http://overpass-api.de/api/interpreter"

    # Query to get all rivers, streams, canals, and drainage
    overpass_query = f"""
    [out:json][timeout:180];
    (
      way["waterway"="river"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["waterway"="stream"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["waterway"="canal"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["waterway"="drain"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["waterway"="ditch"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["waterway"="river"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["waterway"="stream"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["waterway"="drain"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out geom;
    """

    print("Downloading data from Overpass API...")
    print("This may take several minutes...")

    try:
        response = requests.post(overpass_url, data={"data": overpass_query}, timeout=300)
        response.raise_for_status()
        data = response.json()

        print(f"Downloaded {len(data.get('elements', []))} elements")

        # Convert to GeoDataFrame
        features = []

        for element in data.get('elements', []):
            if element['type'] == 'way' and 'geometry' in element:
                coords = [(node['lon'], node['lat']) for node in element['geometry']]
                if len(coords) >= 2:
                    geom = LineString(coords)
                    props = element.get('tags', {})
                    props['osm_id'] = element['id']
                    props['osm_type'] = 'way'
                    features.append({'geometry': geom, 'properties': props})

            elif element['type'] == 'relation' and 'members' in element:
                # Handle relations (complex waterways)
                for member in element['members']:
                    if 'geometry' in member:
                        coords = [(node['lon'], node['lat']) for node in member['geometry']]
                        if len(coords) >= 2:
                            geom = LineString(coords)
                            props = element.get('tags', {})
                            props['osm_id'] = element['id']
                            props['osm_type'] = 'relation'
                            props['member_ref'] = member.get('ref')
                            features.append({'geometry': geom, 'properties': props})

        if not features:
            print("No waterway features found!")
            return None

        # Create GeoDataFrame
        geometries = [f['geometry'] for f in features]
        properties = [f['properties'] for f in features]

        gdf = gpd.GeoDataFrame(properties, geometry=geometries, crs="EPSG:4326")

        print(f"\nCreated GeoDataFrame with {len(gdf)} features")
        print(f"Waterway types: {gdf['waterway'].value_counts().to_dict()}")

        # Save to GeoJSON
        gdf.to_file(output_file, driver='GeoJSON')
        print(f"\nSaved to {output_file}")

        # Also save as shapefile
        shapefile_name = output_file.replace('.geojson', '.shp')
        gdf.to_file(shapefile_name, driver='ESRI Shapefile')
        print(f"Saved to {shapefile_name}")

        return gdf

    except requests.exceptions.Timeout:
        print("Request timed out. Try reducing the bounding box or try again later.")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None


def extract_by_waterway_type(gdf, waterway_type):
    """
    Filter GeoDataFrame by waterway type

    Parameters:
    gdf: GeoDataFrame with waterway data
    waterway_type: 'river', 'stream', or 'canal'
    """
    if gdf is None:
        return None

    filtered = gdf[gdf['waterway'] == waterway_type].copy()
    print(f"\nFound {len(filtered)} {waterway_type}s")

    if len(filtered) > 0 and 'name' in filtered.columns:
        named = filtered[filtered['name'].notna()]
        print(f"Named {waterway_type}s: {len(named)}")
        if len(named) > 0:
            print("Sample names:", list(named['name'].head(10)))

    return filtered


def main():
    # Kerala bounding box (approximate coordinates)
    # Format: (min_lat, min_lon, max_lat, max_lon)
    kerala_bbox = (8.17, 74.85, 12.78, 77.42)

    print("="*60)
    print("OSM Kerala Rivers and Streams Extractor")
    print("="*60)
    print(f"\nBounding box: {kerala_bbox}")
    print("(min_lat, min_lon, max_lat, max_lon)")

    # Download all waterway data
    gdf = download_osm_data(kerala_bbox, "kerala_waterways.geojson")

    if gdf is not None:
        # Extract individual types
        rivers = extract_by_waterway_type(gdf, 'river')
        if rivers is not None and len(rivers) > 0:
            rivers.to_file("kerala_rivers.shp", driver='ESRI Shapefile')
            print(f"Saved rivers to kerala_rivers.shp")

        streams = extract_by_waterway_type(gdf, 'stream')
        if streams is not None and len(streams) > 0:
            streams.to_file("kerala_streams.shp", driver='ESRI Shapefile')
            print(f"Saved streams to kerala_streams.shp")

        canals = extract_by_waterway_type(gdf, 'canal')
        if canals is not None and len(canals) > 0:
            canals.to_file("kerala_canals.shp", driver='ESRI Shapefile')
            print(f"Saved canals to kerala_canals.shp")

        drains = extract_by_waterway_type(gdf, 'drain')
        if drains is not None and len(drains) > 0:
            drains.to_file("kerala_drains.shp", driver='ESRI Shapefile')
            print(f"Saved drains to kerala_drains.shp")

        ditches = extract_by_waterway_type(gdf, 'ditch')
        if ditches is not None and len(ditches) > 0:
            ditches.to_file("kerala_ditches.shp", driver='ESRI Shapefile')
            print(f"Saved ditches to kerala_ditches.shp")

        print("\n" + "="*60)
        print("Extraction complete!")
        print("="*60)
        print("\nFiles created:")
        print("- kerala_waterways.geojson (all waterways)")
        print("- kerala_waterways.shp (all waterways)")
        print("- kerala_rivers.shp (rivers only)")
        print("- kerala_streams.shp (streams only)")
        print("- kerala_canals.shp (canals only)")
        print("- kerala_drains.shp (drains only)")
        print("- kerala_ditches.shp (ditches only)")


if __name__ == "__main__":
    main()

OSM Kerala Rivers and Streams Extractor

Bounding box: (8.17, 74.85, 12.78, 77.42)
(min_lat, min_lon, max_lat, max_lon)
This may take several minutes...
Downloaded 107858 elements

Created GeoDataFrame with 108227 features
Waterway types: {'stream': 92545, 'canal': 9325, 'river': 2603, 'ditch': 1990, 'drain': 1764}

Saved to kerala_waterways.geojson


/tmp/ipython-input-2264995707.py:99: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(shapefile_name, driver='ESRI Shapefile')
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ml' to 'name_ml'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ta' to 'name_ta'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:kn' to 'name_kn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:en' to 'name_en'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:bn' to 'name_bn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normali

Saved to kerala_waterways.shp

Found 2603 rivers
Named rivers: 1522
Sample names: ['Kallada', 'Kallada', 'Amayizhanchan Thodu', 'Kāveri', 'Kaveri', 'Bhavani', 'Geddhehaada Halla River', 'Pykara', 'Muvattupuzha River', 'Kodur River']


/tmp/ipython-input-2264995707.py:153: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  rivers.to_file("kerala_rivers.shp", driver='ESRI Shapefile')
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ml' to 'name_ml'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ta' to 'name_ta'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:kn' to 'name_kn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:en' to 'name_en'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:bn' to 'name_bn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning

Saved rivers to kerala_rivers.shp

Found 92545 streams
Named streams: 14549
Sample names: ['vimmala thode', 'njeliyam kuzhi thodu', 'Chirappurathukadavu Thodu', 'Meenantharayar', 'Puducherry thodu', 'Akkatthodu', 'Ponnozhukum Thodu', 'Pathipaalam - Palakkattuthazham Thodu', 'parappan thod', 'Meenachil thodu']


/tmp/ipython-input-2264995707.py:158: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  streams.to_file("kerala_streams.shp", driver='ESRI Shapefile')
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ml' to 'name_ml'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ta' to 'name_ta'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:kn' to 'name_kn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:en' to 'name_en'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:bn' to 'name_bn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarni

Saved streams to kerala_streams.shp

Found 9325 canals
Named canals: 1923
Sample names: ['Parvathy Puthanar', 'Srambi Kadavu', 'Thottakom thodu', 'Kodappamund Canal', 'Varuna Canal', 'Kuthiathode Canal', 'udayanapuram thodu', 'Lower Bhavani Project Canal', 'Kabini Canal 1', 'Kabini right bank canal']


/tmp/ipython-input-2264995707.py:163: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  canals.to_file("kerala_canals.shp", driver='ESRI Shapefile')
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ml' to 'name_ml'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ta' to 'name_ta'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:kn' to 'name_kn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:en' to 'name_en'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:bn' to 'name_bn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning

Saved canals to kerala_canals.shp

Found 1764 drains
Named drains: 24
Sample names: ['Backwaters', 'Mangalavanam Thodu', 'Amayizhanjan Thodu', 'Backwaters', 'Amayizhanjan Thodu', 'Rameswaram Canal', 'Amayizhanjan Thodu', 'Backwaters', 'Backwaters', 'Backwaters']
Saved drains to kerala_drains.shp

Found 1990 ditchs
Named ditchs: 57
Sample names: ['Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal', 'Pruvannamuzhi-Kakkodi Branch Canal']


/tmp/ipython-input-2264995707.py:168: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  drains.to_file("kerala_drains.shp", driver='ESRI Shapefile')
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ml' to 'name_ml'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:ta' to 'name_ta'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:kn' to 'name_kn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:en' to 'name_en'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'name:bn' to 'name_bn'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning

Saved ditches to kerala_ditches.shp

Extraction complete!

Files created:
- kerala_waterways.geojson (all waterways)
- kerala_waterways.shp (all waterways)
- kerala_rivers.shp (rivers only)
- kerala_streams.shp (streams only)
- kerala_canals.shp (canals only)
- kerala_drains.shp (drains only)
- kerala_ditches.shp (ditches only)
